# Knowledge Graphs and Semantic Technologies -- RDFS tutorial

Here we'll learn the basics of RDFS (RDF Schema) and how to perform basic RDFS reasoning with rdflib (documentation [here](https://rdflib.readthedocs.io/en/stable/)) and owlrl (documentation [here](https://owl-rl.readthedocs.io/en/latest/)).

## Imports

owlrl is a library implementing basic RDFS and OWL reasoning on top of rdflib. We'll install and import its relevant symbols.

In [54]:
import sys
!{sys.executable} -m pip install rdflib owlrl

from rdflib import Graph, RDFS, RDF, URIRef, Namespace, Literal
from owlrl import DeductiveClosure, RDFS_Semantics

zsh:1: no such file or directory: /Users/kim/Desktop/P4/Knowledge


## Loading RDFS graphs

Your file `yourRDF.ttl` already contains a basic Knowledge Graph in RDF with some RDFS semantics

First, we are going to add some RDFS semantics, and inspect the graph as-is; this is also called the "asserted graph"

**Exercise 1** 
1. add additional triples using the RDFS semantics: have a look [here](https://www.w3.org/TR/rdf-schema/), and use domain and range, subPropertyOf, and Class, to say more about the instances in your graph
2. load yourRDF graph
3. print the classes in your graph
4. print the properties of a specific class in yourRDF graph
5. print all instances in yourRDF graph (all objects that have a type) 
6. explain what constitutes a vocabulary in RDF

In [55]:
# Load existing graph
my_namespace_graph = Graph()
my_namespace_graph.parse("data/kim.ttl", format="turtle")

# Define my namespace
KIM = Namespace("https://mynameis/kim#")
my_namespace_graph.namespace_manager.bind("kim", KIM)

# Add triples using store's add method.
my_namespace_graph.add((KIM.Human, RDF.type, RDFS.Class))
my_namespace_graph.add((KIM.Person, RDFS.subClassOf, KIM.Human))

my_namespace_graph.add((KIM.hasParent, RDF.type, RDF.Property))
my_namespace_graph.add((KIM.hasParent, RDFS.domain, KIM.Person))
my_namespace_graph.add((KIM.hasParent, RDFS.range, KIM.Person))

my_namespace_graph.add((KIM.hasMother, RDF.type, RDF.Property))
my_namespace_graph.add((KIM.hasMother, RDFS.subPropertyOf, KIM.hasParent))

my_namespace_graph.serialize(destination="data/kim_combined.ttl", format="turtle")

# Print classes
print("All classes are:")
for classes in my_namespace_graph.subjects(RDF.type, RDFS.Class):
    print(c)

# Print properties
print("All properties are:")
for p in my_namespace_graph.subjects(RDF.type, RDF.Property):
    print(p)

# Print all instances
print("All instances are:")
for s, _, o in my_namespace_graph.triples((None, RDF.type, None)):
    if o not in (RDFS.Class, RDF.Property):
        print(s, "a", o)


All classes are:
https://example.org/Person
https://example.org/Person
https://example.org/Person
https://example.org/Person
https://example.org/Person
All properties are:
https://mynameis/kim#Age
https://mynameis/kim#hasParent
https://mynameis/kim#hasMother
All instances are:


## RDFS inferencing

The inference engine in owlrl is triggered by `DeductiveClosure`, which computes the closure of the graph. This requires us to specify under which semantic regime we want to perform the inference (e.g. what kind of rules under the RDFS, OWL, etc. semantics we want the reasoner to produce derivations on). For RDFS semantics we use `RDFS_Semantics` as parameter. See extra options [here](https://owl-rl.readthedocs.io/en/latest/stubs/owlrl.html#module-owlrl)


**Exercise 2**
1. expand the graph through RDFS semantics inference
2. print how many triples the new graph has
3. print out the triples in your new graph and inspect them. 

In [56]:
# TIP:always look at linked documentation

my_graph = Graph()
my_graph.parse("data/kim.ttl", format="turtle")
print("Triples BEFORE inference:", len(my_graph))

DeductiveClosure(RDFS_Semantics).expand(my_graph)
print("Triples AFTER inference:", len(my_graph))

# 3) Print triples (inspect)
for subj, pred, obj in my_graph:
    print(subj, pred, obj)

Triples BEFORE inference: 11
Triples AFTER inference: 41
https://mynameis/kim#Name http://www.w3.org/2000/01/rdf-schema#subClassOf https://mynameis/kim#Name
https://mynameis/kim#City http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2000/01/rdf-schema#Class
http://www.w3.org/2000/01/rdf-schema#label http://www.w3.org/2000/01/rdf-schema#subPropertyOf http://www.w3.org/2000/01/rdf-schema#label
http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/1999/02/22-rdf-syntax-ns#Property
https://mynameis/kim#Age http://www.w3.org/2000/01/rdf-schema#subPropertyOf https://mynameis/kim#Age
https://mynameis/kim#Name http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2000/01/rdf-schema#Class
https://mynameis/kim#Age http://www.w3.org/2000/01/rdf-schema#label Age
Amsterdam http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2000/01/rdf-schema#Resource
https://mynameis/kim#Amsterdam http://www.w3.or

## The explicit (asserted) graph vs the implicit (derived) graph, and RDF entailment

Asserted triples are those that are explicitly stated, while derived or inferred triples are those that are implicitly stated through the semantics of RDFS. 

**Exercise 3**

1. Write here code to generate a graph that contains **RDFS derived triples only** from yourRDF Knowledge Graph, not the asserted ones. See a clue on rdflib graph algebra [here](https://rdflib.readthedocs.io/en/stable/merging.html)
2. have a look at the inferred graph. Based on the RDFS semantics, explain for each triple the rule that was used to generate it.
3. Explain the concept RDF entailment, and the types of entailment RDFS can produce


In [57]:
kim = Graph()
kim.parse("data/kim.ttl", format="turtle")

inferred = Graph()
inferred.parse("data/kim_combined.ttl", format="turtle")

inferred = inferred - kim
print("RDFS inference generated additional {} triples".format(len(inferred)))
for subj,pred,obj in sorted(inferred.triples((None,None,None))):
    print(subj,pred,obj)

RDFS inference generated additional 7 triples
https://mynameis/kim#Human http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2000/01/rdf-schema#Class
https://mynameis/kim#Person http://www.w3.org/2000/01/rdf-schema#subClassOf https://mynameis/kim#Human
https://mynameis/kim#hasMother http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/1999/02/22-rdf-syntax-ns#Property
https://mynameis/kim#hasMother http://www.w3.org/2000/01/rdf-schema#subPropertyOf https://mynameis/kim#hasParent
https://mynameis/kim#hasParent http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/1999/02/22-rdf-syntax-ns#Property
https://mynameis/kim#hasParent http://www.w3.org/2000/01/rdf-schema#domain https://mynameis/kim#Person
https://mynameis/kim#hasParent http://www.w3.org/2000/01/rdf-schema#range https://mynameis/kim#Person


## Restaurant Assignment - Part 2


**Exercise 4**
1. load ingredients.rdf and recipes.rdf in one graph. The graph contains types of individuals and types of relationships between them. Print all the classes and properties in the combined graph with the namespace `ind` and the `wtm` namespace/vocabulary (`http://purl.org/heals/food/`). 

2. extend the `ind` vocabulary (`http://purl.org/heals/ingredient/`) by creating a hierarchy of ingredients (**hint: http://purl.org/heals/ingredient/CoconutMilk rdf:subClassOf http://purl.org/heals/ingredient/PlantMilk), and make these superclasses human readable by giving them labels**) 
3. do the same for the `wtm` vocabulary: add a hierarchy of recipes as well as a hierarchy of properties (**hint: http://purl.org/heals/food/hasCookingTemperature rdf:subPropertyOf ...) 
4. print the entailed triples as we did in the previous exercise
5. give three examples of how RDF semantics could aid the chefs in your restaurant 
    
6. which properties and classes could you add to the `wtm` and `ind` vocabularies to further describe your recipe and ingredient knowledge graph, aiding the chefs in your restaurant?  


In [58]:
#In order to extend the vocabularies you have to come up with multiple examples that follows the given hierarchy

combined = Graph()
combined.parse("data/ingredients.rdf")
combined.parse("data/recipes.rdf")

ind = Namespace("http://purl.org/heals/ingredient/")
wtm = Namespace("http://purl.org/heals/food/")

# Function to check if they are in the vocabulary
def in_vocab(uri: URIRef, vocab: Namespace) -> bool:
    return str(uri).startswith(str(vocab))

classes = set()
for item in combined.subjects(RDF.type, RDFS.Class):
    if in_vocab(item, ind) or in_vocab(item, wtm):
        classes.add(item)

properties = set()
for property in combined.subjects(RDF.type, RDF.Property):
    if in_vocab(property, ind) or in_vocab(property, wtm):
        properties.add(property)

print("Classes in ind/wtm:")
for c in sorted(classes):
    print(c)

print("\nProperties in ind/wtm:")
for p in sorted(properties):
    print(p)



Classes in ind/wtm:

Properties in ind/wtm:


In [59]:
combined.add((ind.PlantMilk, RDF.type, RDFS.Class))
combined.add((ind.CoconutMilk, RDFS.subClassOf, ind.PlantMilk))
combined.add((ind.PlantMilk, RDFS.label, Literal("Plant milk")))

combined.add((ind.Nut, RDF.type, RDFS.Class))
combined.add((ind.Pecan, RDFS.subClassOf, ind.Nut))
combined.add((ind.Nut, RDFS.label, Literal("Nut")))

<Graph identifier=Nd002d0d953b24194bd8796977baabea3 (<class 'rdflib.graph.Graph'>)>

In [60]:
combined.add((wtm.Dessert, RDF.type, RDFS.Class))
combined.add((wtm.Cake, RDF.type, RDFS.Class))
combined.add((wtm.Cake, RDFS.subClassOf, wtm.Dessert))
combined.add((wtm.Dessert, RDFS.label, Literal("Dessert")))
combined.add((wtm.Cake, RDFS.label, Literal("Cake")))

combined.add((wtm.hasCookingParameter, RDF.type, RDF.Property))
combined.add((wtm.hasCookingTemperature, RDF.type, RDF.Property))
combined.add((wtm.hasCookingTemperature, RDFS.subPropertyOf, wtm.hasCookingParameter))

combined.add((wtm.hasCookingParameter, RDFS.label, Literal("Cooking parameter")))
combined.add((wtm.hasCookingTemperature, RDFS.label, Literal("Cooking temperature")))

<Graph identifier=Nd002d0d953b24194bd8796977baabea3 (<class 'rdflib.graph.Graph'>)>

In [61]:
combined = Graph()
combined.parse("data/ingredients.rdf")
combined.parse("data/recipes.rdf")

combined1 = Graph()
combined1.parse("data/ingredients.rdf")
combined1.parse("data/recipes.rdf")

DeductiveClosure(RDFS_Semantics).expand(combined1)

d = combined1 - combined

print(f"Asserted: {len(combined)}")
print(f"Closure: {len(combined1)}")
print(f"Derived-only: {len(d)}")

for s, p, o in sorted(d):
    if o != RDFS.Resource:
        print(s, p, o)

Asserted: 1299
Closure: 2143
Derived-only: 844
http://purl.org/dc/terms/abstract http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/1999/02/22-rdf-syntax-ns#Property
http://purl.org/dc/terms/abstract http://www.w3.org/2000/01/rdf-schema#subPropertyOf http://purl.org/dc/terms/abstract
http://purl.org/dc/terms/contributor http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/1999/02/22-rdf-syntax-ns#Property
http://purl.org/dc/terms/contributor http://www.w3.org/2000/01/rdf-schema#subPropertyOf http://purl.org/dc/terms/contributor
http://purl.org/dc/terms/description http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/1999/02/22-rdf-syntax-ns#Property
http://purl.org/dc/terms/description http://www.w3.org/2000/01/rdf-schema#subPropertyOf http://purl.org/dc/terms/description
http://purl.org/dc/terms/license http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/1999/02/22-rdf-syntax-ns#Property
http://purl.org/dc/terms/license http://www

## HI ontology exploration

In your project, you will be working with a Hybrid Intelligence (HI) ontology. After acquanting yourself with its structure in the previous tutorial, you will now expand it yourself. Using the tools from the exercises above, perform the following actions:

1. Load the HI ontology from the data folder (hi_ontology.ttl) with RDFlib (don't forget to define a Namespace).
2. Create 3 new subclasses using the rdfs:subClassOf predicate.
3. Create at least one domain and one range restriction using rdfs.
4. Populate each class you've created with at least one new instance each (hint: use rdf:type).




In [62]:
 # 1) Load ontology + define namespace
hi_graph = Graph()
hi_graph.parse("data/hi_ontology.ttl", format="turtle")

HI = Namespace("http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51#")
hi_graph.namespace_manager.bind("hi", HI)

print("Extra subclasses:")
hi_graph.add((HI.HealthMonitoringAgent, RDF.type, RDFS.Class))
hi_graph.add((HI.HealthMonitoringAgent, RDFS.subClassOf, HI.Agent))
hi_graph.add((HI.ChronicCondition, RDF.type, RDFS.Class))
hi_graph.add((HI.ChronicCondition, RDFS.subClassOf, HI.HealthIssue))
hi_graph.add((HI.AgentPulse, RDF.type, HI.HealthMonitoringAgent))
hi_graph.add((HI.DiabetesType2, RDF.type, HI.ChronicCondition))
hi_graph.add((HI.AgentPulse, HI.detectsIssue, HI.DiabetesType2))
hi_graph.add((HI.HealthMonitoringAgent, RDFS.label, Literal("Health monitoring agent")))
hi_graph.add((HI.ChronicCondition, RDFS.label, Literal("Chronic condition")))
hi_graph.add((HI.AgentPulse, RDFS.label, Literal("Agent Pulse")))
hi_graph.add((HI.DiabetesType2, RDFS.label, Literal("Type 2 diabetes")))

# Save
hi_graph.serialize("data/hi_ontology_extended.ttl", format="turtle")
print("Saved to data/hi_ontology_extended.ttl")



Extra subclasses:
Saved to data/hi_ontology_extended.ttl
